In [0]:
%sql
SHOW CATALOGS

In [0]:
%sql
SELECT current_catalog(), current_schema()

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS dev_lab;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dev_lab.bronze;
CREATE SCHEMA IF NOT EXISTS dev_lab.silver;
CREATE SCHEMA IF NOT EXISTS dev_Lab.gold;

In [0]:
%sql
SHOW SCHEMAS IN dev_lab;

C

In [0]:
%sql
-- Criando uma tabela gerenciada
CREATE TABLE IF NOT EXISTS 
    dev_lab.bronze.reading_events(
        event_id STRING,
        user_id STRING,
        book_id STRING,
        event_time TIMESTAMP,
        duration_seconds INT
    )
USING DELTA;

In [0]:
%sql
INSERT INTO dev_lab.bronze.reading_events
VALUES
  ('evt001', 'user01', 'book10',
   TIMESTAMP '2026-10-18 10:00:00', 300),
  ('evt002', 'user02', 'book20',
   TIMESTAMP '2026-10-18 10:01:00', 600);

In [0]:
%sql
SELECT
  book_id,
  COUNT(*) AS total_events,
  SUM(duration_seconds) AS total_seconds
FROM dev_lab.bronze.reading_events
GROUP BY book_id;

Conectar a governança a arquivos reais. Um Volume do Unity Catalog gerencia o acesso a arquivos, como JSON, CSV, checkpoints e outros artefatos. Diferentemente de uma tabela, ele não representa necessriamente dados tabulares.

No Unity Catalog, a hierarquia é:
- Catalogo: dev_lab
- Schema: bronze
- Volume: landing
- Arquivos: eventos JSON que serão ingeridos pelo pipeline

Tabela armazena dados estruturados para consulta. Um Volume gerencia arquivos, como JSON, CSV e outros arquivos de entrada. Vamos usá-lo como área de chegada dos eventos.


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dev_lab.landing;

CREATE VOLUME IF NOT EXISTS
    dev_lab.landing.inbound;

In [0]:
volume_path = "/Volumes/dev_lab/landing/inbound"

eventos = [
    '{"event_id":"evt101","user_id":"user01","book_id":"book10","event_time":"2026-10-18T10:00:00","duration_seconds":300}',
    '{"event_id":"evt102","user_id":"user02","book_id":"book20","event_time":"2026-10-18T10:01:00","duration_seconds":600}',
    '{"event_id":"evt103","user_id":"user01","book_id":"book20","event_time":"2026-10-18T10:02:00","duration_seconds":180}'
]

dbutils.fs.put(
    f"{volume_path}/events_001.json",
    "\n".join(eventos),
    overwrite=True
)

print(dbutils.fs.ls(volume_path))

In [0]:
df = spark.read.json(volume_path)
df.show()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType, IntegerType
)

landing_path = "/Volumes/dev_lab/landing/inbound"

bronze_table = "dev_lab.bronze.reading_events"

checkpoint_path = (
    "/Volumes/dev_lab/landing/inbound/_checkpoints/reading_events"
)

schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("book_id", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("duration_seconds", IntegerType(), True)
])

stream_df = (
    spark.readStream
    .schema(schema)
    .option("maxFilesPerTrigger", 1)
    .json(landing_path)
)

query = (
    stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

In [0]:
%sql
SELECT
    event_id,
    user_id,
    book_id,
    event_time,
    duration_seconds
FROM dev_lab.bronze.reading_events
ORDER BY event_time;

In [0]:
landing_path = "/Volumes/dev_lab/landing/inbound"

novos_eventos = [
    '{"event_id":"evt104","user_id":"user03","book_id":"book10","event_time":"2026-10-18T10:03:00","duration_seconds":420}',
    '{"event_id":"evt105","user_id":"user02","book_id":"book30","event_time":"2026-10-18T10:04:00","duration_seconds":240}'
]

dbutils.fs.put(
    f"{landing_path}/events_002.json",
    "\n".join(novos_eventos),
    overwrite=False
)

display(dbutils.fs.ls(landing_path))